# 04 全データ勾配の集計方式の比較

このNotebookは、全データ平均の cross-entropy 勾配をどの単位で足し、学習データをどこから渡すかを比べる工学的検証である。高速化を共通の学習関数へ入れる前の確認であり、ここだけの実装で測定する。`src/` の学習関数は呼ばずに結果を見てから変えるものとし、このNotebookは `src/` を書き換えない。

比較するのは次の4条件だけである。

- batch size 512、DataLoader
- batch size 4096、DataLoader
- batch size 512、GPU上に学習データを置いた連続走査
- batch size 4096、GPU上に学習データを置いた連続走査

baseline は `bs512_dataloader` である。これは Notebook 03 の勾配集計と同じ単位、同じ DataLoader の使い方に合わせた条件である。

変えないものは、Stage C の初期 checkpoint、データ分割と sample 順、loss、全 50,000 sample の重み、更新する parameter、TT pair の結合、pair の巡回順、learning rate、方向付き SVD、rank 上限、core の書き戻し、validation、seed、dtype と device である。

全データ平均損失は、batch 平均 $L_b$ を sample 数 $n_b$ で重み付ける。

$$
L = \sum_b \frac{n_b}{N} L_b, \qquad N = 50000
$$

$L_b$ は batch の平均 cross-entropy である。最後の batch が小さくても、この定義は変わらない。batch 平均を batch の個数で単純平均しない。

学習率は Notebook 03 の先頭条件に合わせて 0.2 に固定する。TT pair と通常 parameter の両方に同じ 0.2 を使う。0.3 との優劣は、このNotebookでは調べない。


## 1. 設定と比較条件

測定フラグが `False` のセルは、GPU へデータを送らず、CUDA も初期化しない。確認できるのは、設定、依存関係、checkpoint、データ分割、4条件の定義、保存先である。

1局所更新は、6 core の右向き巡回の先頭である。pair index は 0、方向は `right`、開始時の正準中心は 0、local step は 0 である。成功後の中心は 1 になる。Notebook 03 の主実験はこの pair を続けて 8 回更新するが、ここでの1回の測定は、その先頭の1更新だけである。

1往復は、Notebook 03 と同じ pair 順で、各 pair を1回だけ更新する。6 core なので訪問は10回である。右向きが pair 0, 1, 2, 3, 4、そのあと左向きが pair 4, 3, 2, 1, 0 である。Notebook 03 の主実験は各 pair を8回更新するので、そこでの1往復は80更新である。この往復ベンチマークの10更新は、その80更新の代替ではない。

代表時間は、ウォームアップを除いた測定反復の中央値である。1往復の採用に使う代表時間は `one_sweep_wall_seconds` の中央値である。`total_update_seconds` は各局所更新の内部計測 `total_seconds` の合計で、採用判定には使わない。`validation_seconds` は往復後の validation 時間で、速度比較では診断値である。数値比較は、同じ反復番号の baseline と候補を対にする。ウォームアップの数値は保存しない。

合格に使う相対 Frobenius 差の対象は、結合テンソルの勾配、各通常 parameter の勾配、SGD 更新後かつ SVD 前の結合テンソル、SVD 後に再結合した pair、再構成した `fc1` の dense weight、更新後の各通常 parameter、全データ平均 loss である。基準は $10^{-4}$ 未満である。

$$
\frac{\|A-B\|_F}{\|B\|_F} < 10^{-4}
$$

$B$ は baseline、分母は $\|B\|_F$ である。分母が 0、または $10^{-8}$ 未満のときは、この割り算をしない。最大絶対差と Frobenius 差の両方が $10^{-5}$ 未満であることを絶対差の判定として使い、判定方法を記録する。core の要素差は、符号と縮退部分空間の非一意性があるため、不合格条件にしない。特異値、捨てた二乗ノルム、retained rank は別途一致を見る。forward の logits 差は診断値であり、追加の合格線にはしない。

1往復の validation loss の絶対差は $10^{-3}$ 未満、accuracy の絶対差は $0.001$ 未満である。accuracy は 0 から 1 の割合であり、0.001 は 0.1 percentage point である。予測一致率、不一致 sample 数、logits の RMSE、logits の最大絶対差は診断値であり、合格線には加えない。

peak allocated memory は、GPU の総メモリの 80% 未満であることを要求する。reserved memory は別列に記録し、allocated と混ぜない。

採用には、1局所更新の数値基準、1往復の validation 基準、非有限値がないこと、メモリ基準、baseline より短いこと、更新対象と更新順が同じことをすべて要求する。さらに、1往復の代表時間が baseline の 1.10 倍以上短いこと、つまり baseline 時間 / 候補時間 が 1.10 以上であることも、このNotebookの冒頭で決めた採用条件である。これは結果を見てから足した線ではない。合格が複数なら、その中で1往復の代表時間が最短の条件を推奨する。合格がなければ baseline を維持する。最速であることだけでは採用しない。

この検証が通っても、40往復、local step 8、学習率 0.3、別モデルでの安定性は保証しない。


In [5]:
import copy
import csv
import hashlib
import json
import math
import os
import random
import statistics
import struct
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "nn_compression").is_dir():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("src/nn_compression が見つかりません。")

from nn_compression.compression import (
    TTLinear,
    assign_tt_linear_cores,
    module_forward_with_merged_tt_linear_pair,
)
from nn_compression.datasets import (
    get_fashion_mnist_datasets,
    split_fashion_mnist_dataset,
)
from nn_compression.models import MNISTMLP
from nn_compression.training import NonFiniteLocalUpdate, evaluate
from nn_compression.tt_mps import (
    merge_tt_matrix_cores,
    replace_tt_matrix_core_pair,
    split_merged_tt_matrix,
    tt_matrix_canonicality_errors,
    tt_matrix_ranks,
    tt_matrix_to_dense,
)
from nn_compression.utils import find_project_root, get_experiment_dirs, set_seed

# 長い GPU 測定は、それぞれのフラグが True のときだけ始める。
RUN_SINGLE_UPDATE_BENCHMARK = True
RUN_ONE_SWEEP_BENCHMARK = True
# 測定が実行されたときだけ、04 の結果ディレクトリへ書く。
SAVE_RESULTS = True

SEED = 0
OUT_MODES = (4, 2, 2, 2, 4, 4)
IN_MODES = (2, 2, 2, 2, 7, 7)
RANK_CAP = 12
EXPECTED_INITIAL_RANKS = (1, 8, 12, 12, 12, 12, 1)
LAYER_NAME = "fc1"
N_CORES = 6
JOINT_LEARNING_RATE = 0.2
DIAGNOSTIC_BATCH_SIZE = 512
VALIDATION_BATCH_SIZE = 512
SPLIT_LENGTHS = (50_000, 5_000, 5_000)
N_TRAIN = 50_000
WARMUP_REPEATS = 1
MEASUREMENT_REPEATS = 3
REPRESENTATIVE_TIME = (
    "one-sweep adoption uses the median of one_sweep_wall_seconds; "
    "total_update_seconds and validation_seconds are not used for adoption; "
    "warmup, restore, and preload are excluded"
)
SINGLE_UPDATE_PAIR_INDEX = 0
SINGLE_UPDATE_DIRECTION = "right"
SINGLE_UPDATE_CENTER = 0
SINGLE_UPDATE_LOCAL_STEP = 0
# 右向き pair 0..4 のあと、左向き pair 4..0。各 pair の local step は 1 回。
ONE_SWEEP_VISITS = (
    (0, "right"),
    (1, "right"),
    (2, "right"),
    (3, "right"),
    (4, "right"),
    (4, "left"),
    (3, "left"),
    (2, "left"),
    (1, "left"),
    (0, "left"),
)
EXPECTED_CENTERS_AFTER = (1, 2, 3, 4, 5, 4, 3, 2, 1, 0)
EXPECTED_UPDATED_PARAMETERS = (
    "fc1.bias",
    "fc2.weight",
    "fc2.bias",
    "fc3.weight",
    "fc3.bias",
)
BASELINE_CONDITION_ID = "bs512_dataloader"
BENCHMARK_CONDITIONS = (
    {"condition_id": "bs512_dataloader", "batch_size": 512, "data_residency": "dataloader"},
    {"condition_id": "bs4096_dataloader", "batch_size": 4096, "data_residency": "dataloader"},
    {"condition_id": "bs512_gpu_resident", "batch_size": 512, "data_residency": "gpu_resident"},
    {"condition_id": "bs4096_gpu_resident", "batch_size": 4096, "data_residency": "gpu_resident"},
)
RELATIVE_TENSOR_DIFFERENCE_TOLERANCE = 1e-4
RELATIVE_DENOMINATOR_FLOOR = 1e-8
ABSOLUTE_TENSOR_DIFFERENCE_TOLERANCE = 1e-5
ONE_SWEEP_VALIDATION_LOSS_ABSOLUTE_TOLERANCE = 1e-3
ONE_SWEEP_VALIDATION_ACCURACY_ABSOLUTE_TOLERANCE = 0.001
MAXIMUM_PEAK_MEMORY_FRACTION = 0.80
MINIMUM_TIME_IMPROVEMENT_RATIO = 1.10
EXPERIMENT_NAME = "04_full_batch_gradient_optimization_benchmark"
PROTECTED_NAMES = {
    "00_full_batch_two_core_sweep_training",
    "01_tt_initialization_scale_diagnostics",
    "02_full_batch_sweep_update_budget_grid",
    "02_full_batch_sweep_update_budget_grid_smoke",
    "02_full_batch_sweep_update_budget_grid_smoke_bs512",
    "03_joint_learning_rate_02_03_s40_l8",
    "03_joint_learning_rate_02_03_s40_l8_smoke",
}
CHECK_INDICES = (0, 511, 512, 4095, 4096, 49151, 49152, 49663, 49664, 49999)

assert math.prod(OUT_MODES) == 512
assert math.prod(IN_MODES) == 784
assert len(OUT_MODES) == N_CORES
assert JOINT_LEARNING_RATE == 0.2
assert SINGLE_UPDATE_PAIR_INDEX == 0 and SINGLE_UPDATE_DIRECTION == "right"
assert SINGLE_UPDATE_CENTER == 0 and SINGLE_UPDATE_LOCAL_STEP == 0
assert [pair for pair, direction in ONE_SWEEP_VISITS[:5]] == [0, 1, 2, 3, 4]
assert [direction for _, direction in ONE_SWEEP_VISITS[:5]] == ["right"] * 5
assert [pair for pair, direction in ONE_SWEEP_VISITS[5:]] == [4, 3, 2, 1, 0]
assert [direction for _, direction in ONE_SWEEP_VISITS[5:]] == ["left"] * 5
assert len(ONE_SWEEP_VISITS) == 10
assert EXPECTED_CENTERS_AFTER == (1, 2, 3, 4, 5, 4, 3, 2, 1, 0)
assert BENCHMARK_CONDITIONS[0]["condition_id"] == BASELINE_CONDITION_ID
assert [item["condition_id"] for item in BENCHMARK_CONDITIONS] == [
    "bs512_dataloader",
    "bs4096_dataloader",
    "bs512_gpu_resident",
    "bs4096_gpu_resident",
]
assert len({item["condition_id"] for item in BENCHMARK_CONDITIONS}) == 4
assert EXPERIMENT_NAME not in PROTECTED_NAMES
assert EXPERIMENT_NAME.startswith("04_")
assert MINIMUM_TIME_IMPROVEMENT_RATIO == 1.10
assert RELATIVE_TENSOR_DIFFERENCE_TOLERANCE == 1e-4
assert ONE_SWEEP_VALIDATION_ACCURACY_ABSOLUTE_TOLERANCE == 0.001
assert MAXIMUM_PEAK_MEMORY_FRACTION == 0.80

project_root = find_project_root(Path.cwd())
NOTEBOOK_03 = (
    project_root
    / "notebooks"
    / "40_dmrg"
    / "10_fashion_mnist_mlp"
    / "03_joint_learning_rate_02_03_s40_l8.ipynb"
)
INITIAL_CHECKPOINT = (
    project_root
    / "models"
    / "40_dmrg"
    / "10_fashion_mnist_mlp"
    / "01_tt_initialization_scale_diagnostics"
    / "initial_stage_c.pt"
)
TRAINING_MODULE = project_root / "src" / "nn_compression" / "training" / "tt_linear_full_batch_sweep.py"
_, models_dir, results_dir = get_experiment_dirs(
    project_root,
    "40_dmrg",
    "10_fashion_mnist_mlp",
    EXPERIMENT_NAME,
)


def assert_benchmark_output(path):
    resolved = Path(path).resolve()
    allowed = {models_dir.resolve(), results_dir.resolve()}
    if not any(resolved == root or root in resolved.parents for root in allowed):
        raise RuntimeError(f"04 の保存先の外です: {resolved}")
    text = resolved.as_posix()
    for name in PROTECTED_NAMES:
        if name in text:
            raise RuntimeError(f"00〜03 の保存先へ書き込もうとしました: {resolved}")


for directory in (models_dir, results_dir):
    assert_benchmark_output(directory)
    if directory.name in PROTECTED_NAMES or not directory.name.startswith("04_"):
        raise RuntimeError(f"保存先の名前が 04 専用ではありません: {directory}")

print("RUN_SINGLE_UPDATE_BENCHMARK:", RUN_SINGLE_UPDATE_BENCHMARK)
print("RUN_ONE_SWEEP_BENCHMARK:", RUN_ONE_SWEEP_BENCHMARK)
print("SAVE_RESULTS:", SAVE_RESULTS)
print("experiment:", EXPERIMENT_NAME)
print("このセルは CUDA を初期化しません。device は測定を開始するセルで選びます。")
for item in BENCHMARK_CONDITIONS:
    print(item["condition_id"], "batch", item["batch_size"], item["data_residency"])


RUN_SINGLE_UPDATE_BENCHMARK: True
RUN_ONE_SWEEP_BENCHMARK: True
SAVE_RESULTS: True
experiment: 04_full_batch_gradient_optimization_benchmark
このセルは CUDA を初期化しません。device は測定を開始するセルで選びます。
bs512_dataloader batch 512 dataloader
bs4096_dataloader batch 4096 dataloader
bs512_gpu_resident batch 512 gpu_resident
bs4096_gpu_resident batch 4096 gpu_resident


## 2. データ分割、sample 順、Stage C

このセルが比べるのは、DataLoader が返す batch と、同じ sample 順で積んだ CPU tensor の切片である。GPU 上の複製とは、測定を開始したときだけ比べる。

変数 `train_images` と `train_labels` は、seed 0 の学習 50,000 件を subset の順に積んだ float32 画像と int64 ラベルである。checksum は、この CPU tensor のバイト列の SHA-256 である。

期待するのは、batch size 512 と 4096 のどちらでも、先頭から末尾まで欠落と重複がなく、指定した境界の sample が切片と一致することである。checkpoint については、stage C、modes、rank 上限 12、中心 0、初期 rank 列、parameter 名が Notebook 03 と同じであることである。

ここでは GPU へ転送しない。したがって、GPU 上で要素が一致することと、4条件の数値同等性は、このセルだけでは保証できない。test と rank-selection の中身は読まない。


In [6]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def tensor_sha256(tensor):
    array = tensor.detach().contiguous().cpu().numpy()
    return hashlib.sha256(array.tobytes()).hexdigest()


def subset_sha256(subset):
    digest = hashlib.sha256()
    indices = [int(index) for index in subset.indices]
    for index in indices:
        digest.update(struct.pack("<q", index))
    return {
        "n": len(indices),
        "head": indices[:5],
        "tail": indices[-5:],
        "sha256": digest.hexdigest(),
    }


def slice_bounds(n_samples, batch_size):
    if batch_size < 1:
        raise ValueError(f"batch_size は 1 以上である必要があります: {batch_size}")
    bounds = []
    start = 0
    while start < n_samples:
        end = min(start + batch_size, n_samples)
        bounds.append((start, end))
        start = end
    covered = [index for start, end in bounds for index in range(start, end)]
    if covered != list(range(n_samples)):
        raise RuntimeError("batch 切片が sample を過不足なく覆っていません。")
    return bounds


def make_gradient_loader(dataset, batch_size):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=False,
    )


def assert_loader_matches_stack(loader, images, labels, batch_size):
    bounds = slice_bounds(images.shape[0], batch_size)
    seen = 0
    for batch_index, ((start, end), (batch_images, batch_labels)) in enumerate(
        zip(bounds, loader, strict=True)
    ):
        if not torch.equal(batch_images, images[start:end]):
            raise AssertionError(f"batch {batch_index} の画像が CPU stack と一致しません。")
        if not torch.equal(batch_labels, labels[start:end]):
            raise AssertionError(f"batch {batch_index} のラベルが CPU stack と一致しません。")
        seen += int(batch_labels.shape[0])
    if seen != images.shape[0]:
        raise AssertionError(f"DataLoader の sample 数が {images.shape[0]} ではありません: {seen}")
    return bounds


# 分割の再現に使う generator は CPU 上で作る。set_seed は CUDA を初期化するため、測定開始まで呼ばない。
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not INITIAL_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Stage C checkpoint がありません: {INITIAL_CHECKPOINT}")
if not NOTEBOOK_03.is_file():
    raise FileNotFoundError(f"Notebook 03 がありません: {NOTEBOOK_03}")
if not TRAINING_MODULE.is_file():
    raise FileNotFoundError(f"学習モジュールがありません: {TRAINING_MODULE}")

CHECKPOINT_SHA256 = file_sha256(INITIAL_CHECKPOINT)
NOTEBOOK_03_SHA256 = file_sha256(NOTEBOOK_03)
TRAINING_MODULE_SHA256 = file_sha256(TRAINING_MODULE)
try:
    GIT_HEAD = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=project_root,
        text=True,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_HEAD = "unavailable"

fashion_root = project_root / "data" / "FashionMNIST" / "raw"
required_names = [
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte",
]
missing_names = [name for name in required_names if not (fashion_root / name).is_file()]
if missing_names:
    raise FileNotFoundError("Fashion-MNIST がありません: " + ", ".join(missing_names))

full_train, test_dataset = get_fashion_mnist_datasets(project_root / "data", download=False)
train_dataset, early_stopping_dataset, rank_selection_dataset = split_fashion_mnist_dataset(
    full_train,
    SPLIT_LENGTHS,
    seed=SEED,
)
if len(train_dataset) != N_TRAIN:
    raise AssertionError(f"学習 sample 数が {N_TRAIN} ではありません: {len(train_dataset)}")

print("train tensors を CPU 上の subset 順に積みます。")
train_images = torch.empty((N_TRAIN, 1, 28, 28), dtype=torch.float32)
train_labels = torch.empty((N_TRAIN,), dtype=torch.long)
for index in range(N_TRAIN):
    image, label = train_dataset[index]
    train_images[index].copy_(image)
    train_labels[index] = int(label)
IMAGE_SHA256 = tensor_sha256(train_images)
LABEL_SHA256 = tensor_sha256(train_labels)

for batch_size in (512, 4096):
    loader = make_gradient_loader(train_dataset, batch_size)
    bounds = assert_loader_matches_stack(loader, train_images, train_labels, batch_size)
    print(
        "dataloader matches stack",
        "batch",
        batch_size,
        "n_batches",
        len(bounds),
        "last_batch",
        bounds[-1][1] - bounds[-1][0],
    )
for index in CHECK_INDICES:
    image, label = train_dataset[index]
    if not torch.equal(image, train_images[index]) or int(label) != int(train_labels[index]):
        raise AssertionError(f"境界 sample {index} が stack と一致しません。")
print("checked sample indices:", CHECK_INDICES)

validation_loader = make_gradient_loader(early_stopping_dataset, VALIDATION_BATCH_SIZE)
criterion = nn.CrossEntropyLoss()
if criterion.reduction != "mean":
    raise AssertionError(f"criterion.reduction は mean である必要があります: {criterion.reduction}")

SPLIT_IDENTITY = {
    "train": subset_sha256(train_dataset),
    "early_stopping_validation": subset_sha256(early_stopping_dataset),
    "rank_selection_validation": subset_sha256(rank_selection_dataset),
}
print(
    "split lengths:",
    len(train_dataset),
    len(early_stopping_dataset),
    len(rank_selection_dataset),
)
print("test is not scanned:", len(test_dataset))
print("rank-selection validation is not scanned:", len(rank_selection_dataset))
print("image sha256:", IMAGE_SHA256)
print("label sha256:", LABEL_SHA256)


def core_keys(state):
    prefix = f"{LAYER_NAME}.cores."
    return sorted(
        (key for key in state if key.startswith(prefix)),
        key=lambda key: int(key.rsplit(".", 1)[-1]),
    )


def assert_checkpoint_contract(payload):
    if payload.get("layer_name") != LAYER_NAME:
        raise ValueError(f"layer_name が一致しません: {payload.get('layer_name')!r}")
    if tuple(payload.get("out_modes", ())) != OUT_MODES:
        raise ValueError(f"out_modes が一致しません: {payload.get('out_modes')!r}")
    if tuple(payload.get("in_modes", ())) != IN_MODES:
        raise ValueError(f"in_modes が一致しません: {payload.get('in_modes')!r}")
    if int(payload.get("rank_cap", -1)) != RANK_CAP:
        raise ValueError(f"rank_cap が一致しません: {payload.get('rank_cap')!r}")
    if int(payload.get("canonical_center", -1)) != 0:
        raise ValueError(f"canonical_center は 0 である必要があります: {payload.get('canonical_center')!r}")
    if payload.get("stage") != "C":
        raise ValueError(f"stage C ではありません: {payload.get('stage')!r}")
    if int(payload.get("seed", -1)) != SEED:
        raise ValueError(f"seed が一致しません: {payload.get('seed')!r}")


def build_model_on_cpu(path):
    payload = torch.load(Path(path), map_location="cpu", weights_only=False)
    assert_checkpoint_contract(payload)
    state = payload["state_dict"]
    keys = core_keys(state)
    if len(keys) != N_CORES:
        raise ValueError(f"core 数が {N_CORES} ではありません: {len(keys)}")
    cores = [state[key].detach().to(dtype=torch.float32, device="cpu") for key in keys]
    ranks = tt_matrix_ranks(cores)
    if tuple(ranks) != EXPECTED_INITIAL_RANKS:
        raise ValueError(f"初期 rank 列が一致しません: {tuple(ranks)}")
    errors = tt_matrix_canonicality_errors(cores, OUT_MODES, IN_MODES, 0)
    if errors["left"]:
        raise ValueError("site 0 正準のはずが、左側の直交誤差があります。")
    ortho_error = max(float(item) for item in errors["right"])
    if ortho_error > 1e-4:
        raise ValueError(f"site 0 の右直交誤差が大きすぎます: {ortho_error}")
    model = MNISTMLP()
    model.fc1 = TTLinear(cores, OUT_MODES, IN_MODES, bias=state[f"{LAYER_NAME}.bias"].detach())
    own = dict(model.named_parameters())
    if set(own) != set(state):
        raise ValueError("checkpoint とモデルの parameter 名が一致しません。")
    with torch.no_grad():
        for key, value in state.items():
            if key.startswith(f"{LAYER_NAME}.cores.") or key == f"{LAYER_NAME}.bias":
                continue
            own[key].copy_(value)
    for key, value in state.items():
        current = model.state_dict()[key]
        if current.dtype != torch.float32:
            raise TypeError(f"{key} の dtype が float32 ではありません: {current.dtype}")
        if not torch.equal(current.cpu(), value.detach().to(dtype=torch.float32, device="cpu")):
            raise ValueError(f"初期 state が checkpoint と一致しません: {key}")
    for core in model.fc1.cores:
        core.requires_grad_(False)
    parameter_names = tuple(name for name, _parameter in model.named_parameters())
    frozen = tuple(name for name in parameter_names if name.startswith("fc1.cores."))
    updated = tuple(name for name in parameter_names if name not in frozen)
    if frozen != tuple(f"fc1.cores.{index}" for index in range(N_CORES)):
        raise AssertionError(f"凍結 core が一致しません: {frozen}")
    if updated != EXPECTED_UPDATED_PARAMETERS:
        raise AssertionError(f"更新対象 parameter が一致しません: {updated}")
    if any(parameter.requires_grad for parameter in model.fc1.cores):
        raise AssertionError("TT core の requires_grad が残っています。")
    return model, ortho_error, parameter_names


cpu_model, initial_ortho_error, parameter_names = build_model_on_cpu(INITIAL_CHECKPOINT)
if file_sha256(INITIAL_CHECKPOINT) != CHECKPOINT_SHA256:
    raise RuntimeError("checkpoint を読んだだけで内容が変わりました。")
buffer_names = tuple(name for name, _buffer in cpu_model.named_buffers())
print("initial ranks:", EXPECTED_INITIAL_RANKS)
print("site-0 right orthogonality max error:", initial_ortho_error)
print("parameters:", parameter_names)
print("buffers:", buffer_names)
print("canonical center from checkpoint: 0")
del cpu_model


train tensors を CPU 上の subset 順に積みます。
dataloader matches stack batch 512 n_batches 98 last_batch 336
dataloader matches stack batch 4096 n_batches 13 last_batch 848
checked sample indices: (0, 511, 512, 4095, 4096, 49151, 49152, 49663, 49664, 49999)
split lengths: 50000 5000 5000
test is not scanned: 10000
rank-selection validation is not scanned: 5000
image sha256: 6a1fd0602db5a1066210a55375d18f9b12115269e7c6759f64504735f020d4d5
label sha256: 6ead6f3f16028865d36fd3c8625eba751820bd8031265a3be03e206136a3c96c
initial ranks: (1, 8, 12, 12, 12, 12, 1)
site-0 right orthogonality max error: 9.382572443428217e-07
parameters: ('fc1.bias', 'fc1.cores.0', 'fc1.cores.1', 'fc1.cores.2', 'fc1.cores.3', 'fc1.cores.4', 'fc1.cores.5', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias')
buffers: ()
canonical center from checkpoint: 0


## 3. 差の定義と、1回の局所更新

このセルは測定を始めない。数値差の判定と、既存の結合、SVD、書き戻しを呼ぶ測定関数を定義する。

`tensor_difference` の戻り値は、最大絶対差、Frobenius 差、相対 Frobenius 差、baseline のノルム、候補のノルム、判定方法、その判定での合否である。相対差の分母が $10^{-8}$ 未満のとき、`decision_method` は絶対差であり、小さい定数で割った相対差は作らない。

`run_local_update` の時間は、データ取得、CPU から GPU への転送、forward / backward と勾配集計、結合テンソルの SGD 更新、方向付き SVD、core の書き戻しと候補 forward に分ける。合計時間は、これらの壁時計時間であり、状態の復元と結果 tensor の CPU への複製は含めない。CUDA では各区間の前後で同期する。

GPU 事前配置の batch は、常駐 tensor の連続切片だけを使う。その区間の転送時間は 0 として記録する。DataLoader 条件は、現行の集計と同じく batch ごとに CPU から GPU へ送る。

この関数定義だけでは、Fashion-MNIST 上の4条件が数値的に一致するとは言えない。


In [7]:
def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def tensor_difference(baseline, candidate):
    baseline = baseline.detach().to(dtype=torch.float32, device="cpu")
    candidate = candidate.detach().to(dtype=torch.float32, device="cpu")
    finite = bool(torch.isfinite(baseline).all() and torch.isfinite(candidate).all())
    if tuple(baseline.shape) != tuple(candidate.shape):
        return {
            "max_abs": None,
            "frobenius": None,
            "relative_frobenius": None,
            "baseline_norm": None,
            "candidate_norm": None,
            "decision_method": "shape_mismatch",
            "passed": False,
            "finite": finite,
        }
    difference = candidate - baseline
    max_abs = float(difference.abs().max()) if difference.numel() else 0.0
    frobenius = float(torch.linalg.vector_norm(difference))
    baseline_norm = float(torch.linalg.vector_norm(baseline))
    candidate_norm = float(torch.linalg.vector_norm(candidate))
    if (not math.isfinite(baseline_norm)) or baseline_norm < RELATIVE_DENOMINATOR_FLOOR:
        # 分母が極端に小さいときは割らない。絶対差だけで判定し、そのことを記録する。
        method = "absolute_because_baseline_norm_is_small"
        relative = None
        passed = (
            finite
            and max_abs < ABSOLUTE_TENSOR_DIFFERENCE_TOLERANCE
            and frobenius < ABSOLUTE_TENSOR_DIFFERENCE_TOLERANCE
        )
    else:
        method = "relative_frobenius"
        relative = frobenius / baseline_norm
        passed = finite and relative < RELATIVE_TENSOR_DIFFERENCE_TOLERANCE
    return {
        "max_abs": max_abs,
        "frobenius": frobenius,
        "relative_frobenius": relative,
        "baseline_norm": baseline_norm,
        "candidate_norm": candidate_norm,
        "decision_method": method,
        "passed": passed,
        "finite": finite,
    }


def _assert_difference_rules():
    identical = tensor_difference(torch.ones(2, 2), torch.ones(2, 2))
    if identical["decision_method"] != "relative_frobenius" or not identical["passed"]:
        raise AssertionError("一致する tensor が相対差で合格になりません。")
    small = tensor_difference(torch.tensor([1.0, 0.0]), torch.tensor([1.0 + 1e-6, 0.0]))
    large = tensor_difference(torch.tensor([1.0, 0.0]), torch.tensor([1.0 + 1e-3, 0.0]))
    if not small["passed"] or large["passed"]:
        raise AssertionError("相対差 1e-4 未満の判定が期待と違います。")
    zeros = tensor_difference(torch.zeros(2), torch.zeros(2))
    if zeros["decision_method"] != "absolute_because_baseline_norm_is_small" or not zeros["passed"]:
        raise AssertionError("ゼロ tensor が絶対差判定になっていません。")
    zero_vs_large = tensor_difference(torch.zeros(2), torch.full((2,), 1e-4))
    if zero_vs_large["decision_method"] != "absolute_because_baseline_norm_is_small":
        raise AssertionError("小さい分母を相対差に回しています。")
    if zero_vs_large["passed"] or zero_vs_large["relative_frobenius"] is not None:
        raise AssertionError("絶対差で不合格になる差を、割り算で処理しています。")


_assert_difference_rules()
print("difference rules checked on CPU tensors. This is not the Fashion-MNIST comparison.")


def capture_state(model):
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}


def restore_state(model, saved):
    # capture_state は CPU に退避する。assign_tt_linear_cores は layer と同じ device を要求するので、
    # core も dense parameter と同様に、戻す先の device / dtype へ移してから書く。
    target_device = model.fc1.cores[0].device
    target_dtype = model.fc1.cores[0].dtype
    assign_tt_linear_cores(
        model.fc1,
        [
            saved[f"fc1.cores.{index}"]
            .to(device=target_device, dtype=target_dtype)
            .clone()
            for index in range(N_CORES)
        ],
        requires_grad=False,
    )
    named = dict(model.named_parameters())
    for name, value in saved.items():
        if name.startswith("fc1.cores."):
            continue
        parameter = named[name]
        with torch.no_grad():
            parameter.copy_(value.to(device=parameter.device, dtype=parameter.dtype))
        parameter.requires_grad_(True)
    for name, parameter in model.named_parameters():
        current = parameter.detach().cpu()
        if not torch.equal(current, saved[name]):
            raise RuntimeError(f"初期 state へ戻せません: {name}")


def iter_resident_batches(images, labels, batch_size):
    for start, end in slice_bounds(images.shape[0], batch_size):
        synchronize(images.device)
        started = time.perf_counter()
        batch_images = images[start:end]
        batch_labels = labels[start:end]
        synchronize(images.device)
        fetch_seconds = time.perf_counter() - started
        yield batch_images, batch_labels, fetch_seconds, 0.0


def iter_loader_batches(loader, device):
    iterator = iter(loader)
    while True:
        synchronize(device)
        started = time.perf_counter()
        try:
            images, labels = next(iterator)
        except StopIteration:
            break
        synchronize(device)
        fetch_seconds = time.perf_counter() - started
        synchronize(device)
        moved_at = time.perf_counter()
        images = images.to(device, non_blocking=False)
        labels = labels.to(device, non_blocking=False)
        synchronize(device)
        h2d_seconds = time.perf_counter() - moved_at
        yield images, labels, fetch_seconds, h2d_seconds


def diagnostic_forward(model, images, labels):
    flags = [(module, module.training) for module in model.modules()]
    try:
        model.eval()
        with torch.no_grad():
            logits = model(images)
            loss = criterion(logits, labels)
    finally:
        for module, training in flags:
            module.training = training
    return logits, loss


def commit_pair(model, candidate, new_cores, dense_candidates):
    assign_tt_linear_cores(
        candidate.fc1,
        [core.detach() for core in model.fc1.cores],
        requires_grad=False,
    )
    source = dict(model.named_parameters())
    for name, parameter in candidate.named_parameters():
        if name.startswith("fc1.cores."):
            continue
        with torch.no_grad():
            parameter.copy_(source[name].detach())
        parameter.requires_grad_(True)
    assign_tt_linear_cores(candidate.fc1, new_cores, requires_grad=False)
    candidate_parameters = dict(candidate.named_parameters())
    for name, value in dense_candidates.items():
        parameter = candidate_parameters[name]
        with torch.no_grad():
            parameter.copy_(value.to(parameter.device))
    return candidate


def write_candidate_back(model, candidate):
    assign_tt_linear_cores(
        model.fc1,
        [core.detach() for core in candidate.fc1.cores],
        requires_grad=False,
    )
    source = dict(candidate.named_parameters())
    for name, parameter in model.named_parameters():
        if name.startswith("fc1.cores."):
            continue
        with torch.no_grad():
            parameter.copy_(source[name].detach())
        parameter.requires_grad_(True)


def run_local_update(
    model,
    candidate,
    batches,
    *,
    pair_index,
    direction,
    center,
    diagnostic_images,
    diagnostic_labels,
    device,
    n_samples,
):
    if direction not in {"right", "left"}:
        raise ValueError(f"direction が不正です: {direction!r}")
    if center not in {pair_index, pair_index + 1}:
        raise ValueError(f"center={center} は pair {pair_index} の外です。")
    for name, parameter in model.named_parameters():
        parameter.requires_grad_(not name.startswith("fc1.cores."))
        parameter.grad = None

    timings = {"data_fetch": 0.0, "h2d": 0.0, "forward_backward": 0.0}
    synchronize(device)
    total_started = time.perf_counter()
    synchronize(device)
    section = time.perf_counter()
    merged = (
        merge_tt_matrix_cores(model.fc1.cores[pair_index], model.fc1.cores[pair_index + 1])
        .detach()
        .clone()
        .requires_grad_(True)
    )
    synchronize(device)
    # 最新 core から T を作る時間は、勾配を計算する区間に含める。
    timings["forward_backward"] += time.perf_counter() - section

    seen = 0
    n_batches = 0
    last_batch_size = 0
    loss_total = torch.zeros((), device=device, dtype=torch.float32)
    for images, labels, fetch_seconds, h2d_seconds in batches:
        timings["data_fetch"] += fetch_seconds
        timings["h2d"] += h2d_seconds
        n_b = int(labels.shape[0])
        if images.shape[0] != n_b:
            raise ValueError("images と labels の batch 次元が一致しません。")
        synchronize(device)
        section = time.perf_counter()
        logits = module_forward_with_merged_tt_linear_pair(
            model,
            images,
            LAYER_NAME,
            pair_index,
            merged,
        )
        loss_b = criterion(logits, labels)
        if not bool(torch.isfinite(loss_b).all()) or not bool(torch.isfinite(logits).all()):
            raise NonFiniteLocalUpdate("batch loss または logits が有限ではありません。")
        # 平均 loss を batch 数で平均せず、sample 数の重み (n_b / N) を掛ける。
        weighted = loss_b * (n_b / n_samples)
        loss_total = loss_total + weighted.detach()
        weighted.backward()
        synchronize(device)
        timings["forward_backward"] += time.perf_counter() - section
        seen += n_b
        n_batches += 1
        last_batch_size = n_b
    if seen != n_samples:
        raise RuntimeError(f"処理 sample 数 {seen} が {n_samples} と一致しません。")
    if merged.grad is None:
        raise RuntimeError("結合テンソルの勾配がありません。")

    synchronize(device)
    section = time.perf_counter()
    grad_t = merged.grad.detach().clone()
    dense_grads = {}
    for name, parameter in model.named_parameters():
        if name.startswith("fc1.cores.") or not parameter.requires_grad:
            continue
        if parameter.grad is None:
            raise RuntimeError(f"{name} の勾配がありません。")
        dense_grads[name] = parameter.grad.detach().clone()
    if tuple(dense_grads) != EXPECTED_UPDATED_PARAMETERS:
        raise RuntimeError(f"更新対象の勾配が一致しません: {tuple(dense_grads)}")
    if not bool(torch.isfinite(grad_t).all()):
        raise NonFiniteLocalUpdate("結合テンソルの勾配が有限ではありません。")
    for name, gradient in dense_grads.items():
        if not bool(torch.isfinite(gradient).all()):
            raise NonFiniteLocalUpdate(f"{name} の勾配が有限ではありません。")
    synchronize(device)
    timings["forward_backward"] += time.perf_counter() - section

    synchronize(device)
    section = time.perf_counter()
    t_candidate = merged.detach() - JOINT_LEARNING_RATE * grad_t
    dense_candidates = {}
    for name, parameter in model.named_parameters():
        if name not in dense_grads:
            continue
        updated = parameter.detach() - JOINT_LEARNING_RATE * dense_grads[name]
        if not bool(torch.isfinite(updated).all()):
            raise NonFiniteLocalUpdate(f"{name} の更新候補が有限ではありません。")
        dense_candidates[name] = updated
    if not bool(torch.isfinite(t_candidate).all()):
        raise NonFiniteLocalUpdate("更新後の結合テンソルが有限ではありません。")
    synchronize(device)
    timings["merged_update"] = time.perf_counter() - section

    synchronize(device)
    section = time.perf_counter()
    split = split_merged_tt_matrix(
        t_candidate.detach(),
        max_bond=RANK_CAP,
        direction=direction,
    )
    synchronize(device)
    timings["svd"] = time.perf_counter() - section
    if not bool(torch.isfinite(split.singular_values).all()):
        raise NonFiniteLocalUpdate("特異値が有限ではありません。")
    if int(split.retained_rank) > RANK_CAP:
        raise RuntimeError(f"retained rank が上限を超えています: {split.retained_rank}")

    synchronize(device)
    section = time.perf_counter()
    new_cores = replace_tt_matrix_core_pair(
        [core.detach() for core in model.fc1.cores],
        pair_index,
        split.left_core,
        split.right_core,
    )
    commit_pair(model, candidate, new_cores, dense_candidates)
    if diagnostic_images.device != device:
        diagnostic_images = diagnostic_images.to(device, non_blocking=False)
        diagnostic_labels = diagnostic_labels.to(device, non_blocking=False)
    diagnostic_logits, diagnostic_loss = diagnostic_forward(
        candidate,
        diagnostic_images,
        diagnostic_labels,
    )
    if not bool(torch.isfinite(diagnostic_logits).all()) or not bool(torch.isfinite(diagnostic_loss).all()):
        raise NonFiniteLocalUpdate("SVD 後の候補 forward が有限ではありません。")
    write_candidate_back(model, candidate)
    synchronize(device)
    timings["writeback"] = time.perf_counter() - section
    synchronize(device)
    total_seconds = time.perf_counter() - total_started
    new_center = pair_index + 1 if direction == "right" else pair_index

    cores = [core.detach().cpu().clone() for core in model.fc1.cores]
    reconstructed_pair = merge_tt_matrix_cores(cores[pair_index], cores[pair_index + 1]).cpu()
    reconstructed_weight = tt_matrix_to_dense(cores, OUT_MODES, IN_MODES).cpu()
    dense_parameters = {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if name in dense_candidates
    }
    return {
        "pair_index": pair_index,
        "direction": direction,
        "center_before": center,
        "center_after": new_center,
        "full_batch_loss": loss_total.detach().cpu(),
        "grad_t": grad_t.detach().cpu(),
        "dense_grads": {name: value.detach().cpu() for name, value in dense_grads.items()},
        "t_candidate": t_candidate.detach().cpu(),
        "singular_values": split.singular_values.detach().cpu(),
        "cores": cores,
        "reconstructed_pair": reconstructed_pair,
        "reconstructed_weight": reconstructed_weight,
        "dense_parameters": dense_parameters,
        "retained_rank": int(split.retained_rank),
        "discarded_squared_norm": split.discarded_squared_norm.detach().cpu(),
        "diagnostic_logits": diagnostic_logits.detach().cpu(),
        "samples_seen": seen,
        "n_batches": n_batches,
        "last_batch_size": last_batch_size,
        "timings": timings,
        "total_seconds": total_seconds,
        "finite": True,
    }


single_update_runs = []
one_sweep_runs = []
difference_rows = []
measurement_device = None
measurement_gpu_name = None
measurement_total_memory_bytes = None


difference rules checked on CPU tensors. This is not the Fashion-MNIST comparison.


## 4. 1局所更新の測定

測定するのは、同じ Stage C、同じ pair 0、同じ右向き、同じ学習率 0.2、同じ rank 上限 12 からの1更新である。4条件の差は、勾配を足す batch の大きさと、その batch を DataLoader から渡すか GPU 上の連続切片から渡すかだけである。

各反復の前に、モデルをその条件の開始時の checkpoint 状態へ戻す。前の条件の重みは渡さない。ウォームアップは1回で、CSV には残さない。測定は3回で、代表時間はその合計時間の中央値である。

保存する数値は、全データ loss、勾配、更新後の結合テンソル、特異値、SVD 後の core、再構成 weight、通常 parameter、retained rank、捨てた二乗ノルム、peak memory、有限性である。

フラグが `False` のときは、このセルは GPU 測定を始めない。フラグを後から `True` にしても、このセルの定義確認だけでは40往復の速度や安定性は分からない。


In [8]:
def begin_measurement():
    global measurement_device, measurement_gpu_name, measurement_total_memory_bytes
    # Notebook 03 と同じ set_seed。CUDA 初期化を伴うため、測定フラグが立ったときだけ呼ぶ。
    set_seed(SEED)
    if not torch.cuda.is_available():
        raise RuntimeError("この測定は CUDA が必要です。CPU へ落としての代替測定はしません。")
    measurement_device = torch.device("cuda")
    measurement_gpu_name = torch.cuda.get_device_name(measurement_device)
    measurement_total_memory_bytes = int(
        torch.cuda.get_device_properties(measurement_device).total_memory
    )
    return measurement_device


def assert_gpu_copy_matches(gpu_images, gpu_labels):
    for index in CHECK_INDICES:
        if not torch.equal(gpu_images[index].detach().cpu(), train_images[index]):
            raise AssertionError(f"GPU 上の画像が CPU stack と一致しません: index {index}")
        if int(gpu_labels[index].detach().cpu()) != int(train_labels[index]):
            raise AssertionError(f"GPU 上のラベルが CPU stack と一致しません: index {index}")
    if tensor_sha256(gpu_images) != IMAGE_SHA256 or tensor_sha256(gpu_labels) != LABEL_SHA256:
        raise AssertionError("GPU 常駐 tensor の checksum が CPU stack と一致しません。")


def prepare_condition(spec, device):
    model, _ortho_error, _names = build_model_on_cpu(INITIAL_CHECKPOINT)
    model.to(device)
    for core in model.fc1.cores:
        core.requires_grad_(False)
    saved = capture_state(model)
    restore_state(model, saved)
    resident = None
    preload_seconds = 0.0
    if spec["data_residency"] == "gpu_resident":
        synchronize(device)
        started = time.perf_counter()
        gpu_images = train_images.to(device, non_blocking=False).contiguous()
        gpu_labels = train_labels.to(device, non_blocking=False).contiguous()
        synchronize(device)
        preload_seconds = time.perf_counter() - started
        assert_gpu_copy_matches(gpu_images, gpu_labels)
        resident = (gpu_images, gpu_labels)
    elif spec["data_residency"] != "dataloader":
        raise ValueError(f"data_residency が不正です: {spec['data_residency']!r}")
    return model, saved, resident, preload_seconds


def make_batches(spec, resident, device):
    if spec["data_residency"] == "dataloader":
        loader = make_gradient_loader(train_dataset, int(spec["batch_size"]))
        return iter_loader_batches(loader, device)
    gpu_images, gpu_labels = resident
    return iter_resident_batches(gpu_images, gpu_labels, int(spec["batch_size"]))


def diagnostic_tensors(spec, resident, device):
    if spec["data_residency"] == "gpu_resident":
        gpu_images, gpu_labels = resident
        return gpu_images[:DIAGNOSTIC_BATCH_SIZE], gpu_labels[:DIAGNOSTIC_BATCH_SIZE]
    return train_images[:DIAGNOSTIC_BATCH_SIZE], train_labels[:DIAGNOSTIC_BATCH_SIZE]


def untouched_state_ok(before, after, pair_index):
    changed = {f"fc1.cores.{pair_index}", f"fc1.cores.{pair_index + 1}"}
    for name, value in before.items():
        if name in changed or name in EXPECTED_UPDATED_PARAMETERS:
            continue
        if not torch.equal(value, after[name].detach().cpu()):
            return False
    return True


def memory_snapshot(device):
    if device.type != "cuda":
        raise RuntimeError("GPU memory は CUDA の測定でだけ記録します。")
    allocated = int(torch.cuda.max_memory_allocated(device))
    reserved = int(torch.cuda.max_memory_reserved(device))
    total = int(measurement_total_memory_bytes)
    return {
        "peak_allocated_bytes": allocated,
        "peak_reserved_bytes": reserved,
        "total_gpu_memory_bytes": total,
        "peak_memory_fraction": allocated / total,
    }


def run_one_local_measurement(spec, model, saved, resident, device):
    synchronize(device)
    restore_started = time.perf_counter()
    restore_state(model, saved)
    candidate = copy.deepcopy(model)
    synchronize(device)
    restore_seconds = time.perf_counter() - restore_started
    images, labels = diagnostic_tensors(spec, resident, device)
    torch.cuda.reset_peak_memory_stats(device)
    result = run_local_update(
        model,
        candidate,
        make_batches(spec, resident, device),
        pair_index=SINGLE_UPDATE_PAIR_INDEX,
        direction=SINGLE_UPDATE_DIRECTION,
        center=SINGLE_UPDATE_CENTER,
        diagnostic_images=images,
        diagnostic_labels=labels,
        device=device,
        n_samples=N_TRAIN,
    )
    after = capture_state(model)
    result["untouched_state_unchanged"] = untouched_state_ok(
        saved,
        after,
        SINGLE_UPDATE_PAIR_INDEX,
    )
    result["restore_seconds"] = restore_seconds
    result["memory"] = memory_snapshot(device)
    result["parameter_names"] = tuple(result["dense_parameters"])
    return result


def condition_single_update(spec, device):
    print("single update", spec["condition_id"])
    if file_sha256(INITIAL_CHECKPOINT) != CHECKPOINT_SHA256:
        raise RuntimeError("測定開始時に Stage C checkpoint の内容が変わっています。")
    model, saved, resident, preload_seconds = prepare_condition(spec, device)
    runs = []
    try:
        for _warmup_index in range(WARMUP_REPEATS):
            try:
                run_one_local_measurement(spec, model, saved, resident, device)
            except NonFiniteLocalUpdate as exc:
                print("warmup non-finite:", exc)
            print("warmup finished and discarded", spec["condition_id"])
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        return [
            {
                "spec": spec,
                "repeat_index": 0,
                "result": {"status": "out_of_memory", "finite": False, "message": str(exc)},
                "preload_seconds": preload_seconds,
            }
        ]
    for repeat_index in range(MEASUREMENT_REPEATS):
        try:
            result = run_one_local_measurement(spec, model, saved, resident, device)
            result["status"] = "ok"
        except NonFiniteLocalUpdate as exc:
            result = {"status": "non_finite", "finite": False, "message": str(exc)}
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            result = {"status": "out_of_memory", "finite": False, "message": str(exc)}
            runs.append({"spec": spec, "repeat_index": repeat_index, "result": result, "preload_seconds": preload_seconds})
            break
        runs.append(
            {
                "spec": spec,
                "repeat_index": repeat_index,
                "result": result,
                "preload_seconds": preload_seconds,
            }
        )
        print(spec["condition_id"], "repeat", repeat_index, result["status"])
    del model, resident
    torch.cuda.empty_cache()
    return runs


if not RUN_SINGLE_UPDATE_BENCHMARK:
    print("1局所更新ベンチマークは開始しません。")
else:
    device = begin_measurement()
    single_update_runs = []
    for spec in BENCHMARK_CONDITIONS:
        single_update_runs.extend(condition_single_update(spec, device))
    print("single update repeats recorded:", len(single_update_runs))


single update bs512_dataloader
warmup finished and discarded bs512_dataloader
bs512_dataloader repeat 0 ok
bs512_dataloader repeat 1 ok
bs512_dataloader repeat 2 ok
single update bs4096_dataloader
warmup finished and discarded bs4096_dataloader
bs4096_dataloader repeat 0 ok
bs4096_dataloader repeat 1 ok
bs4096_dataloader repeat 2 ok
single update bs512_gpu_resident
warmup finished and discarded bs512_gpu_resident
bs512_gpu_resident repeat 0 ok
bs512_gpu_resident repeat 1 ok
bs512_gpu_resident repeat 2 ok
single update bs4096_gpu_resident
warmup finished and discarded bs4096_gpu_resident
bs4096_gpu_resident repeat 0 ok
bs4096_gpu_resident repeat 1 ok
bs4096_gpu_resident repeat 2 ok
single update repeats recorded: 12


## 5. 1往復の測定

1往復は、上で定義した10訪問を、同じモデル上で順に1回ずつ行うものである。各条件は再び Stage C から作り、1往復の途中では初期状態へ戻さない。反復の間では初期状態へ戻す。

記録するのは、10更新の合計時間、1更新あたりの平均と中央値、各更新の全データ loss、`evaluate` による validation loss と accuracy、最終 core、最終の通常 parameter、最終の再構成 weight、peak allocated / reserved memory、retained rank と捨てた二乗ノルムの履歴、有限性である。

validation accuracy の差が 0.001 未満でも、どの sample の予測が違ったかは分からない。そのため、予測一致率、不一致数、logits RMSE、logits 最大絶対差を診断として残す。これらに新しい合格線は置かない。

`total_update_seconds` は、各局所更新の内部で測った `total_seconds` の合計である。局所更新の計測終了後に行う診断 tensor の CPU 転送や再構成、次の pair へ進む Python 処理は含まない。

`one_sweep_wall_seconds` は、初期状態の復元が終わったあと、10回の局所更新を始める直前から、10回目の局所更新と、その後の CPU 転送・再構成までを、同期した wall-clock 時間で測った値である。データ取得、転送、forward / backward、SGD、SVD、書き戻し、局所更新後の診断用 CPU 転送と再構成、次の pair までの Python 処理を含む。checkpoint への復元と、GPU への学習データ事前配置は含めない。

`validation_seconds` は、10回の局所更新が終わったあとの `evaluate` と、validation logits・prediction の収集だけを測る。採用判定の速度比較には使わない。

`unattributed_sweep_overhead_seconds` は `one_sweep_wall_seconds - total_update_seconds` である。計測誤差でごく小さい負値になっても、ゼロへ丸めない。

採用判定は、学習データを準備した後の定常的な1往復処理時間を比較する。GPU 事前配置は1条件につき一度だけ必要な初期化コストであり、`preload_seconds` として別に記録する。この判断は、3,200回の局所更新を行う長時間実験を想定している。短い処理では preload 時間を無視できない。

フラグが `False` のときは実行しない。1往復が速くても、40往復の所要時間や数値安定性は分からない。


In [9]:
def validation_predictions(model, device):
    flags = [(module, module.training) for module in model.modules()]
    logits_chunks = []
    prediction_chunks = []
    try:
        model.eval()
        with torch.no_grad():
            for images, labels in validation_loader:
                images = images.to(device, non_blocking=False)
                logits = model(images)
                logits_chunks.append(logits.detach().cpu())
                prediction_chunks.append(torch.argmax(logits, dim=1).detach().cpu())
    finally:
        for module, training in flags:
            module.training = training
    return torch.cat(logits_chunks), torch.cat(prediction_chunks)


def run_one_sweep_measurement(spec, model, saved, resident, device):
    synchronize(device)
    restore_started = time.perf_counter()
    restore_state(model, saved)
    candidate = copy.deepcopy(model)
    synchronize(device)
    restore_seconds = time.perf_counter() - restore_started
    center = 0
    steps = []
    torch.cuda.reset_peak_memory_stats(device)
    # 復元と事前配置のあと。10回の局所更新とその後処理だけを囲む。
    synchronize(device)
    wall_started = time.perf_counter()
    for pair_index, direction in ONE_SWEEP_VISITS:
        images, labels = diagnostic_tensors(spec, resident, device)
        step = run_local_update(
            model,
            candidate,
            make_batches(spec, resident, device),
            pair_index=pair_index,
            direction=direction,
            center=center,
            diagnostic_images=images,
            diagnostic_labels=labels,
            device=device,
            n_samples=N_TRAIN,
        )
        if step["center_after"] != EXPECTED_CENTERS_AFTER[len(steps)]:
            raise RuntimeError(
                f"中心の移動が pair 順と一致しません: {step['center_after']}"
            )
        center = step["center_after"]
        steps.append(step)
    if center != 0:
        raise RuntimeError(f"1往復後の中心が 0 ではありません: {center}")
    synchronize(device)
    one_sweep_wall_seconds = time.perf_counter() - wall_started
    synchronize(device)
    validation_started = time.perf_counter()
    validation_loss, validation_accuracy = evaluate(model, validation_loader, criterion, device)
    logits, predictions = validation_predictions(model, device)
    synchronize(device)
    validation_seconds = time.perf_counter() - validation_started
    durations = [step["total_seconds"] for step in steps]
    total_update_seconds = float(sum(durations))
    # 内部計測との差。負の計測誤差もそのまま残す。
    unattributed_sweep_overhead_seconds = one_sweep_wall_seconds - total_update_seconds
    final_cores = steps[-1]["cores"]
    return {
        "status": "ok",
        "finite": all(step["finite"] for step in steps) and math.isfinite(validation_loss) and math.isfinite(validation_accuracy),
        "steps": steps,
        "total_update_seconds": total_update_seconds,
        "one_sweep_wall_seconds": one_sweep_wall_seconds,
        "validation_seconds": validation_seconds,
        "unattributed_sweep_overhead_seconds": unattributed_sweep_overhead_seconds,
        "mean_update_seconds": float(statistics.fmean(durations)),
        "median_update_seconds": float(statistics.median(durations)),
        "validation_loss": float(validation_loss),
        "validation_accuracy": float(validation_accuracy),
        "logits": logits,
        "predictions": predictions,
        "final_cores": final_cores,
        "final_dense": steps[-1]["dense_parameters"],
        "final_weight": tt_matrix_to_dense(final_cores, OUT_MODES, IN_MODES).cpu(),
        "retained_ranks": [step["retained_rank"] for step in steps],
        "discarded_squared_norms": [float(step["discarded_squared_norm"]) for step in steps],
        "centers_after": [step["center_after"] for step in steps],
        "samples_seen": [step["samples_seen"] for step in steps],
        "losses": [float(step["full_batch_loss"]) for step in steps],
        "restore_seconds": restore_seconds,
        "memory": memory_snapshot(device),
        "final_center": center,
    }


def condition_one_sweep(spec, device):
    print("one sweep", spec["condition_id"])
    if file_sha256(INITIAL_CHECKPOINT) != CHECKPOINT_SHA256:
        raise RuntimeError("測定開始時に Stage C checkpoint の内容が変わっています。")
    model, saved, resident, preload_seconds = prepare_condition(spec, device)
    runs = []
    try:
        for _warmup_index in range(WARMUP_REPEATS):
            try:
                run_one_sweep_measurement(spec, model, saved, resident, device)
            except NonFiniteLocalUpdate as exc:
                print("warmup non-finite:", exc)
            print("warmup finished and discarded", spec["condition_id"])
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        return [
            {
                "spec": spec,
                "repeat_index": 0,
                "result": {"status": "out_of_memory", "finite": False, "message": str(exc)},
                "preload_seconds": preload_seconds,
            }
        ]
    for repeat_index in range(MEASUREMENT_REPEATS):
        try:
            result = run_one_sweep_measurement(spec, model, saved, resident, device)
        except NonFiniteLocalUpdate as exc:
            result = {"status": "non_finite", "finite": False, "message": str(exc)}
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            result = {"status": "out_of_memory", "finite": False, "message": str(exc)}
            runs.append({"spec": spec, "repeat_index": repeat_index, "result": result, "preload_seconds": preload_seconds})
            break
        runs.append(
            {
                "spec": spec,
                "repeat_index": repeat_index,
                "result": result,
                "preload_seconds": preload_seconds,
            }
        )
        print(
            spec["condition_id"],
            "repeat",
            repeat_index,
            result["status"],
            "wall_seconds",
            result.get("one_sweep_wall_seconds"),
            "internal_sum_seconds",
            result.get("total_update_seconds"),
        )
    del model, resident
    torch.cuda.empty_cache()
    return runs


if not RUN_ONE_SWEEP_BENCHMARK:
    print("1往復ベンチマークは開始しません。")
else:
    if measurement_device is None:
        begin_measurement()
    one_sweep_runs = []
    for spec in BENCHMARK_CONDITIONS:
        one_sweep_runs.extend(condition_one_sweep(spec, measurement_device))
    print("one sweep repeats recorded:", len(one_sweep_runs))


one sweep bs512_dataloader
warmup finished and discarded bs512_dataloader
bs512_dataloader repeat 0 ok wall_seconds 15.396805600001244 internal_sum_seconds 15.38312230000156
bs512_dataloader repeat 1 ok wall_seconds 18.06779810000444 internal_sum_seconds 18.05137920001289
bs512_dataloader repeat 2 ok wall_seconds 17.20869010000024 internal_sum_seconds 17.192989300005138
one sweep bs4096_dataloader
warmup finished and discarded bs4096_dataloader
bs4096_dataloader repeat 0 ok wall_seconds 16.675540500000352 internal_sum_seconds 16.660611500017694
bs4096_dataloader repeat 1 ok wall_seconds 16.594914999994216 internal_sum_seconds 16.580133200011915
bs4096_dataloader repeat 2 ok wall_seconds 16.844209200004116 internal_sum_seconds 16.82999369998288
one sweep bs512_gpu_resident
warmup finished and discarded bs512_gpu_resident
bs512_gpu_resident repeat 0 ok wall_seconds 2.5978173000039533 internal_sum_seconds 2.5833558000013
bs512_gpu_resident repeat 1 ok wall_seconds 2.5599493000045186 inter

## 6. 数値差、メモリ、採用判定

baseline は `bs512_dataloader` である。1局所更新では、同じ反復番号の tensor 差を合格線と比較する。1往復では、validation loss の絶対差が $10^{-3}$ 未満、accuracy の絶対差が $0.001$ 未満であることを、すべての測定反復に要求する。accuracy の 0.001 は 0.1 percentage point である。

peak allocated bytes / 総 GPU memory が 0.80 未満であることも要求する。reserved は見ながら、この比の分子には使わない。

採用候補は、数値、validation、有限性、メモリ、更新順、baseline より短いこと、時間比 1.10 以上をすべて満たす条件だけである。時間比、baseline より短いか、最低改善率 1.10 かは、`one_sweep_wall_seconds` の中央値で計算する。`total_update_seconds` の中央値は内訳として残し、採用には使わない。`validation_seconds` も診断値である。その中で `one_sweep_wall_seconds` の中央値が最短のものを推奨する。該当がなければ baseline を維持する。

測定フラグがオフのとき、このセルは判定を完了扱いにせず、結果ファイルも書かない。ファイルを書いたあとでも、40往復へそのまま使ってよいことにはならない。


In [10]:
def add_difference(scope, condition_id, repeat_index, quantity, role, baseline_tensor, candidate_tensor):
    compared = tensor_difference(baseline_tensor, candidate_tensor)
    difference_rows.append(
        {
            "scope": scope,
            "condition_id": condition_id,
            "repeat_index": repeat_index,
            "baseline_condition_id": BASELINE_CONDITION_ID,
            "quantity": quantity,
            "role": role,
            "max_abs": compared["max_abs"],
            "frobenius": compared["frobenius"],
            "relative_frobenius": compared["relative_frobenius"],
            "baseline_norm": compared["baseline_norm"],
            "candidate_norm": compared["candidate_norm"],
            "decision_method": compared["decision_method"],
            "passed": compared["passed"],
            "finite": compared["finite"],
        }
    )
    return compared


def runs_by_key(runs):
    found = {}
    for item in runs:
        found[(item["spec"]["condition_id"], item["repeat_index"])] = item
    return found


def single_update_pass(condition_id):
    if condition_id == BASELINE_CONDITION_ID:
        return True
    found = runs_by_key(single_update_runs)
    checks = []
    for repeat_index in range(MEASUREMENT_REPEATS):
        baseline = found.get((BASELINE_CONDITION_ID, repeat_index))
        candidate = found.get((condition_id, repeat_index))
        if baseline is None or candidate is None:
            return False
        base = baseline["result"]
        other = candidate["result"]
        if base.get("status") != "ok" or other.get("status") != "ok":
            return False
        if not other["untouched_state_unchanged"] or other["samples_seen"] != N_TRAIN:
            return False
        if other["parameter_names"] != EXPECTED_UPDATED_PARAMETERS:
            return False
        if other["retained_rank"] != base["retained_rank"]:
            return False
        pairs = [
            ("full_batch_loss", base["full_batch_loss"], other["full_batch_loss"], "pass_criterion"),
            ("grad_t", base["grad_t"], other["grad_t"], "pass_criterion"),
            ("t_candidate", base["t_candidate"], other["t_candidate"], "pass_criterion"),
            ("reconstructed_pair", base["reconstructed_pair"], other["reconstructed_pair"], "pass_criterion"),
            ("reconstructed_weight", base["reconstructed_weight"], other["reconstructed_weight"], "pass_criterion"),
            ("singular_values", base["singular_values"], other["singular_values"], "pass_criterion"),
            ("discarded_squared_norm", base["discarded_squared_norm"], other["discarded_squared_norm"], "pass_criterion"),
            ("diagnostic_logits", base["diagnostic_logits"], other["diagnostic_logits"], "diagnostic"),
        ]
        for name in EXPECTED_UPDATED_PARAMETERS:
            pairs.append((f"dense_grad:{name}", base["dense_grads"][name], other["dense_grads"][name], "pass_criterion"))
            pairs.append(
                (
                    f"dense_parameter:{name}",
                    base["dense_parameters"][name],
                    other["dense_parameters"][name],
                    "pass_criterion",
                )
            )
        for index, (left, right) in enumerate(zip(base["cores"], other["cores"], strict=True)):
            pairs.append((f"core:{index}", left, right, "diagnostic"))
        for quantity, left, right, role in pairs:
            compared = add_difference("single_update", condition_id, repeat_index, quantity, role, left, right)
            if role == "pass_criterion":
                checks.append(compared["passed"] and compared["finite"])
    return bool(checks) and all(checks)


def prediction_diagnostics(baseline_logits, baseline_predictions, logits, predictions):
    disagreement = predictions != baseline_predictions
    logit_difference = tensor_difference(baseline_logits, logits)
    return {
        "prediction_agreement": float((~disagreement).to(torch.float32).mean()),
        "n_prediction_disagreements": int(disagreement.sum()),
        "logits_rmse": None
        if logit_difference["frobenius"] is None
        else logit_difference["frobenius"] / math.sqrt(logits.numel()),
        "logits_max_abs": logit_difference["max_abs"],
    }


def sweep_validation_pass(condition_id):
    if condition_id == BASELINE_CONDITION_ID:
        return True
    found = runs_by_key(one_sweep_runs)
    decisions = []
    for repeat_index in range(MEASUREMENT_REPEATS):
        baseline = found.get((BASELINE_CONDITION_ID, repeat_index))
        candidate = found.get((condition_id, repeat_index))
        if baseline is None or candidate is None:
            return False
        base = baseline["result"]
        other = candidate["result"]
        if base.get("status") != "ok" or other.get("status") != "ok":
            return False
        if other["centers_after"] != list(EXPECTED_CENTERS_AFTER):
            return False
        if any(seen != N_TRAIN for seen in other["samples_seen"]):
            return False
        loss_gap = abs(other["validation_loss"] - base["validation_loss"])
        accuracy_gap = abs(other["validation_accuracy"] - base["validation_accuracy"])
        decisions.append(
            other["finite"]
            and loss_gap < ONE_SWEEP_VALIDATION_LOSS_ABSOLUTE_TOLERANCE
            and accuracy_gap < ONE_SWEEP_VALIDATION_ACCURACY_ABSOLUTE_TOLERANCE
        )
        add_difference(
            "one_sweep",
            condition_id,
            repeat_index,
            "final_reconstructed_weight",
            "diagnostic",
            base["final_weight"],
            other["final_weight"],
        )
    return bool(decisions) and all(decisions)


def schedule_ok(condition_id):
    found = runs_by_key(one_sweep_runs)
    visits = [pair for pair, _direction in ONE_SWEEP_VISITS]
    for repeat_index in range(MEASUREMENT_REPEATS):
        item = found.get((condition_id, repeat_index))
        if item is None or item["result"].get("status") != "ok":
            return False
        result = item["result"]
        if result["centers_after"] != list(EXPECTED_CENTERS_AFTER):
            return False
        if [step["pair_index"] for step in result["steps"]] != visits:
            return False
        if [step["direction"] for step in result["steps"]] != [direction for _pair, direction in ONE_SWEEP_VISITS]:
            return False
    return True


def representative_seconds(runs, field):
    values = []
    for repeat_index in range(MEASUREMENT_REPEATS):
        item = runs_by_key(runs).get((field[0], repeat_index))
        if item is None or item["result"].get("status") != "ok":
            return None
        values.append(float(item["result"][field[1]]))
    return float(statistics.median(values))


def memory_pass(runs, condition_id):
    fractions = []
    for repeat_index in range(MEASUREMENT_REPEATS):
        item = runs_by_key(runs).get((condition_id, repeat_index))
        if item is None or item["result"].get("status") != "ok":
            return False
        fractions.append(float(item["result"]["memory"]["peak_memory_fraction"]))
    return max(fractions) < MAXIMUM_PEAK_MEMORY_FRACTION


def finite_pass(runs, condition_id):
    for repeat_index in range(MEASUREMENT_REPEATS):
        item = runs_by_key(runs).get((condition_id, repeat_index))
        if item is None or not item["result"].get("finite"):
            return False
    return True


single_ready = len(single_update_runs) > 0
sweep_ready = len(one_sweep_runs) > 0
if not single_ready or not sweep_ready:
    print("測定が揃っていないため、採用判定はしません。")
    print("single update recorded:", single_ready, "one sweep recorded:", sweep_ready)
    recommendation = "未測定"
    adoption_rows = []
else:
    difference_rows = []
    adoption_rows = []
    baseline_single = representative_seconds(
        single_update_runs,
        (BASELINE_CONDITION_ID, "total_seconds"),
    )
    baseline_wall = representative_seconds(
        one_sweep_runs,
        (BASELINE_CONDITION_ID, "one_sweep_wall_seconds"),
    )
    for spec in BENCHMARK_CONDITIONS:
        condition_id = spec["condition_id"]
        single_seconds = representative_seconds(single_update_runs, (condition_id, "total_seconds"))
        # 採用に使うのは 10 更新全体の wall-clock 中央値。内部合計は内訳だけ。
        wall_seconds = representative_seconds(one_sweep_runs, (condition_id, "one_sweep_wall_seconds"))
        total_update_median = representative_seconds(one_sweep_runs, (condition_id, "total_update_seconds"))
        ratio = None
        if baseline_wall not in (None, 0) and wall_seconds not in (None, 0):
            ratio = baseline_wall / wall_seconds
        row = {
            "condition_id": condition_id,
            "single_update_median_seconds": single_seconds,
            "one_sweep_median_seconds": wall_seconds,
            "one_sweep_wall_median_seconds": wall_seconds,
            "total_update_median_seconds": total_update_median,
            "time_ratio_baseline_over_candidate": ratio,
            "single_update_numerical_pass": single_update_pass(condition_id),
            "one_sweep_validation_pass": sweep_validation_pass(condition_id),
            "finite_pass": finite_pass(single_update_runs, condition_id) and finite_pass(one_sweep_runs, condition_id),
            "memory_pass": memory_pass(single_update_runs, condition_id) and memory_pass(one_sweep_runs, condition_id),
            "same_schedule": schedule_ok(condition_id),
            "faster_than_baseline": condition_id != BASELINE_CONDITION_ID and ratio is not None and ratio > 1.0,
            "meets_minimum_improvement": condition_id != BASELINE_CONDITION_ID
            and ratio is not None
            and ratio >= MINIMUM_TIME_IMPROVEMENT_RATIO,
        }
        row["adoption_candidate"] = condition_id != BASELINE_CONDITION_ID and all(
            (
                row["single_update_numerical_pass"],
                row["one_sweep_validation_pass"],
                row["finite_pass"],
                row["memory_pass"],
                row["same_schedule"],
                row["faster_than_baseline"],
                row["meets_minimum_improvement"],
            )
        )
        adoption_rows.append(row)
        print(condition_id, "ratio", ratio, "adoption", row["adoption_candidate"])
    candidates = [row for row in adoption_rows if row["adoption_candidate"]]
    if not candidates:
        recommendation = "合格条件がないため、baseline の batch size 512 と DataLoader を維持する。"
    else:
        chosen = min(candidates, key=lambda row: row["one_sweep_wall_median_seconds"])
        recommendation = (
            f"{chosen['condition_id']} を推奨候補にする。"
            f" 1往復の代表時間の比 baseline/candidate は {chosen['time_ratio_baseline_over_candidate']:.4f}。"
        )
    print(recommendation)


def csv_value(value):
    if value is None:
        return ""
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, float):
        return repr(value)
    return value


def write_csv(path, rows, fieldnames):
    assert_benchmark_output(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({key: csv_value(row.get(key)) for key in fieldnames})
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def single_update_table():
    rows = []
    for item in single_update_runs:
        spec = item["spec"]
        result = item["result"]
        row = {
            "condition_id": spec["condition_id"],
            "repeat_index": item["repeat_index"],
            "batch_size": spec["batch_size"],
            "data_residency": spec["data_residency"],
            "status": result.get("status"),
            "finite": result.get("finite"),
            "preload_seconds": item["preload_seconds"],
        }
        if result.get("status") == "ok":
            timings = result["timings"]
            row.update(
                {
                    "pair_index": result["pair_index"],
                    "direction": result["direction"],
                    "center_before": result["center_before"],
                    "center_after": result["center_after"],
                    "data_fetch_seconds": timings["data_fetch"],
                    "h2d_transfer_seconds": timings["h2d"],
                    "forward_backward_seconds": timings["forward_backward"],
                    "merged_update_seconds": timings["merged_update"],
                    "svd_seconds": timings["svd"],
                    "writeback_validation_seconds": timings["writeback"],
                    "segments_sum_seconds": sum(timings.values()),
                    "total_seconds": result["total_seconds"],
                    "restore_seconds": result["restore_seconds"],
                    "full_batch_loss": float(result["full_batch_loss"]),
                    "retained_rank": result["retained_rank"],
                    "discarded_squared_norm": float(result["discarded_squared_norm"]),
                    "samples_seen": result["samples_seen"],
                    "n_batches": result["n_batches"],
                    "last_batch_size": result["last_batch_size"],
                    "untouched_state_unchanged": result["untouched_state_unchanged"],
                    "updated_parameter_names": ",".join(result["parameter_names"]),
                    **result["memory"],
                }
            )
        rows.append(row)
    return rows


def one_sweep_table():
    baseline = runs_by_key(one_sweep_runs)
    rows = []
    for item in one_sweep_runs:
        spec = item["spec"]
        result = item["result"]
        row = {
            "condition_id": spec["condition_id"],
            "repeat_index": item["repeat_index"],
            "batch_size": spec["batch_size"],
            "data_residency": spec["data_residency"],
            "status": result.get("status"),
            "finite": result.get("finite"),
            "preload_seconds": item["preload_seconds"],
        }
        if result.get("status") != "ok":
            rows.append(row)
            continue
        base = baseline.get((BASELINE_CONDITION_ID, item["repeat_index"]))
        if base is not None and base["result"].get("status") == "ok":
            diagnostics = prediction_diagnostics(
                base["result"]["logits"],
                base["result"]["predictions"],
                result["logits"],
                result["predictions"],
            )
        else:
            diagnostics = {
                "prediction_agreement": None,
                "n_prediction_disagreements": None,
                "logits_rmse": None,
                "logits_max_abs": None,
            }
        row.update(
            {
                "n_updates": len(result["steps"]),
                "total_update_seconds": result["total_update_seconds"],
                "one_sweep_wall_seconds": result["one_sweep_wall_seconds"],
                "validation_seconds": result["validation_seconds"],
                "unattributed_sweep_overhead_seconds": result["unattributed_sweep_overhead_seconds"],
                "mean_update_seconds": result["mean_update_seconds"],
                "median_update_seconds": result["median_update_seconds"],
                "step_total_seconds": ",".join(repr(step["total_seconds"]) for step in result["steps"]),
                "step_full_batch_losses": ",".join(repr(loss) for loss in result["losses"]),
                "validation_loss": result["validation_loss"],
                "validation_accuracy": result["validation_accuracy"],
                "retained_ranks": ",".join(str(rank) for rank in result["retained_ranks"]),
                "discarded_squared_norms": ",".join(repr(value) for value in result["discarded_squared_norms"]),
                "centers_after": ",".join(str(value) for value in result["centers_after"]),
                "samples_seen": ",".join(str(value) for value in result["samples_seen"]),
                "final_center": result["final_center"],
                "restore_seconds": result["restore_seconds"],
                **diagnostics,
                **result["memory"],
            }
        )
        rows.append(row)
    return rows


def benchmark_config():
    return {
        "notebook": "04_full_batch_gradient_optimization_benchmark.ipynb",
        "notebook_03": str(NOTEBOOK_03),
        "notebook_03_sha256": NOTEBOOK_03_SHA256,
        "git_head": GIT_HEAD,
        "checkpoint_path": str(INITIAL_CHECKPOINT),
        "checkpoint_sha256": CHECKPOINT_SHA256,
        "training_module_path": str(TRAINING_MODULE),
        "training_module_sha256": TRAINING_MODULE_SHA256,
        "split_identity": SPLIT_IDENTITY,
        "image_sha256": IMAGE_SHA256,
        "label_sha256": LABEL_SHA256,
        "gpu_name": measurement_gpu_name,
        "cuda_version": torch.version.cuda,
        "torch_version": torch.__version__,
        "dtype": "torch.float32",
        "seed": SEED,
        "joint_learning_rate": JOINT_LEARNING_RATE,
        "conditions": list(BENCHMARK_CONDITIONS),
        "single_update": {
            "pair_index": SINGLE_UPDATE_PAIR_INDEX,
            "direction": SINGLE_UPDATE_DIRECTION,
            "center": SINGLE_UPDATE_CENTER,
            "local_step": SINGLE_UPDATE_LOCAL_STEP,
        },
        "one_sweep_visits": [{"pair_index": pair, "direction": direction} for pair, direction in ONE_SWEEP_VISITS],
        "tolerances": {
            "relative_tensor_difference": RELATIVE_TENSOR_DIFFERENCE_TOLERANCE,
            "relative_denominator_floor": RELATIVE_DENOMINATOR_FLOOR,
            "absolute_tensor_difference": ABSOLUTE_TENSOR_DIFFERENCE_TOLERANCE,
            "one_sweep_validation_loss": ONE_SWEEP_VALIDATION_LOSS_ABSOLUTE_TOLERANCE,
            "one_sweep_validation_accuracy": ONE_SWEEP_VALIDATION_ACCURACY_ABSOLUTE_TOLERANCE,
            "maximum_peak_memory_fraction": MAXIMUM_PEAK_MEMORY_FRACTION,
            "minimum_time_improvement_ratio": MINIMUM_TIME_IMPROVEMENT_RATIO,
        },
        "warmup_repeats": WARMUP_REPEATS,
        "measurement_repeats": MEASUREMENT_REPEATS,
        "representative_time": REPRESENTATIVE_TIME,
        "adoption_time_field": "one_sweep_wall_median_seconds",
        "time_definitions": {
            "total_update_seconds": "sum of per-update total_seconds measured inside run_local_update",
            "one_sweep_wall_seconds": "synced wall clock around the 10 local updates, including post-timer CPU transfers and reconstructions",
            "validation_seconds": "evaluate plus validation logits and predictions after the sweep wall clock",
            "unattributed_sweep_overhead_seconds": "one_sweep_wall_seconds - total_update_seconds, not clamped",
            "preload_seconds": "one-time GPU residency copy, excluded from one_sweep_wall_seconds and the adoption ratio",
        },
        "numerical_pairing": "same repeat_index against bs512_dataloader",
        "checked_at": datetime.now().astimezone().isoformat(),
        "src_training_function_was_not_modified": True,
    }


if single_ready and sweep_ready and SAVE_RESULTS:
    config = benchmark_config()
    summary = {
        "recommendation": recommendation,
        "adoption_rows": adoption_rows,
        "representative_time": REPRESENTATIVE_TIME,
        "adoption_time_field": "one_sweep_wall_median_seconds",
        "minimum_time_improvement_ratio": MINIMUM_TIME_IMPROVEMENT_RATIO,
    }
    config_path = results_dir / "benchmark_config.json"
    summary_path = results_dir / "benchmark_summary.json"
    single_path = results_dir / "single_update_benchmark.csv"
    sweep_path = results_dir / "one_sweep_benchmark.csv"
    difference_path = results_dir / "numerical_differences.csv"
    for path in (config_path, summary_path, single_path, sweep_path, difference_path):
        assert_benchmark_output(path)
    config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    write_csv(
        single_path,
        single_update_table(),
        [
            "condition_id",
            "repeat_index",
            "batch_size",
            "data_residency",
            "status",
            "finite",
            "pair_index",
            "direction",
            "center_before",
            "center_after",
            "data_fetch_seconds",
            "h2d_transfer_seconds",
            "forward_backward_seconds",
            "merged_update_seconds",
            "svd_seconds",
            "writeback_validation_seconds",
            "segments_sum_seconds",
            "total_seconds",
            "restore_seconds",
            "preload_seconds",
            "full_batch_loss",
            "retained_rank",
            "discarded_squared_norm",
            "samples_seen",
            "n_batches",
            "last_batch_size",
            "untouched_state_unchanged",
            "updated_parameter_names",
            "peak_allocated_bytes",
            "peak_reserved_bytes",
            "total_gpu_memory_bytes",
            "peak_memory_fraction",
        ],
    )
    write_csv(
        sweep_path,
        one_sweep_table(),
        [
            "condition_id",
            "repeat_index",
            "batch_size",
            "data_residency",
            "status",
            "finite",
            "n_updates",
            "total_update_seconds",
            "one_sweep_wall_seconds",
            "validation_seconds",
            "unattributed_sweep_overhead_seconds",
            "mean_update_seconds",
            "median_update_seconds",
            "step_total_seconds",
            "step_full_batch_losses",
            "validation_loss",
            "validation_accuracy",
            "prediction_agreement",
            "n_prediction_disagreements",
            "logits_rmse",
            "logits_max_abs",
            "retained_ranks",
            "discarded_squared_norms",
            "centers_after",
            "samples_seen",
            "final_center",
            "restore_seconds",
            "preload_seconds",
            "peak_allocated_bytes",
            "peak_reserved_bytes",
            "total_gpu_memory_bytes",
            "peak_memory_fraction",
        ],
    )
    write_csv(
        difference_path,
        difference_rows,
        [
            "scope",
            "condition_id",
            "repeat_index",
            "baseline_condition_id",
            "quantity",
            "role",
            "max_abs",
            "frobenius",
            "relative_frobenius",
            "baseline_norm",
            "candidate_norm",
            "decision_method",
            "passed",
            "finite",
        ],
    )
    print("saved", results_dir)
elif not SAVE_RESULTS:
    print("SAVE_RESULTS が False のため、結果ファイルは書きません。")
else:
    print("測定前なので、結果ファイルは書きません。")


bs512_dataloader ratio 1.0 adoption False
bs4096_dataloader ratio 1.0319719531729648 adoption False
bs512_gpu_resident ratio 6.624288051347588 adoption True
bs4096_gpu_resident ratio 10.48549534173493 adoption True
bs4096_gpu_resident を推奨候補にする。 1往復の代表時間の比 baseline/candidate は 10.4855。
saved D:\dev\nn-compression-svd-dmrg\results\40_dmrg\10_fashion_mnist_mlp\04_full_batch_gradient_optimization_benchmark


## 7. 実行手順と、この結果で言えないこと

測定するときは、Notebook 03 の学習が GPU を使っていないことを確認してから、設定セルのフラグを変える。

1. `RUN_SINGLE_UPDATE_BENCHMARK` を `True` にする。
2. `RUN_ONE_SWEEP_BENCHMARK` を `True` にする。
3. `SAVE_RESULTS` は `True` のままでよい。
4. 先頭から実行する。各条件は Stage C から独立に作られる。

結果は `results/40_dmrg/10_fashion_mnist_mlp/04_full_batch_gradient_optimization_benchmark/` に書かれる。一時 checkpoint が必要になった場合の置き場は `models/40_dmrg/10_fashion_mnist_mlp/04_full_batch_gradient_optimization_benchmark/` であり、00〜03 のディレクトリには書かない。

1局所更新と1往復の合格は、40往復、local step 8、学習率 0.3、他のモデルでの採用を意味しない。共通の学習関数への反映は、保存した CSV と JSON を見てから別に決める。
